# 11 — Multimodal Evaluation and Hallucination

## What
Evaluating VLMs requires specialised benchmarks that test visual grounding, not just language fluency. The key failure mode is *multimodal hallucination* — models describe objects or attributes that are not present in the image, drawing on language priors rather than visual evidence.

## Why
Perplexity and BLEU are inadequate for VLMs — a model can produce fluent, grammatically correct descriptions of images that don't match what's actually shown. Hallucination in medical imaging (describing findings that aren't there) or autonomous driving (claiming the road is clear) has direct safety consequences.

## Community context
MME, MMBench, and SEED-Bench are comprehensive VLM benchmarks covering perception, reasoning, and knowledge. POPE (Polling-based Object Probing Evaluation) specifically tests object hallucination by asking yes/no questions about objects that may or may not be in the image. HallusionBench tests both visual and knowledge hallucination. LLaVA-Bench measures instruction-following quality.

In [ ]:
# VLM hallucination detection: POPE-style object probing
import numpy as np

class POPEEvaluator:
    """
    POPE: ask VLM 'Is there a [object] in the image?'
    Half positive (object present), half negative (object absent).
    Measures hallucination rate as False Positive Rate on negatives.
    """
    def __init__(self):
        self.questions = []   # (question, gt_present)
        self.predictions = []  # (predicted_present,)

    def add_question(self, object_name, gt_present, model_response):
        self.questions.append((object_name, gt_present))
        # Parse model response as yes/no
        pred_present = 'yes' in model_response.lower()
        self.predictions.append(pred_present)

    def evaluate(self):
        tp = fp = tn = fn = 0
        for (obj, gt), pred in zip(self.questions, self.predictions):
            if gt and pred:     tp += 1
            elif not gt and pred: fp += 1  # hallucination!
            elif gt and not pred: fn += 1
            else:               tn += 1
        accuracy  = (tp+tn) / len(self.questions)
        precision = tp / (tp+fp+1e-10)
        recall    = tp / (tp+fn+1e-10)
        f1        = 2*precision*recall / (precision+recall+1e-10)
        hall_rate = fp / (fp+tn+1e-10)  # False Positive Rate = hallucination
        return {
            'accuracy':  round(accuracy, 4),
            'precision': round(precision, 4),
            'recall':    round(recall, 4),
            'f1':        round(f1, 4),
            'hallucination_rate': round(hall_rate, 4),
            'n_hallucinations': fp,
            'n_questions': len(self.questions),
        }

# Simulate two models: one grounded, one language-prior-heavy
np.random.seed(42)

# Ground truth: 50 objects present, 50 absent
objects_present = ['cat','car','tree','building','person','dog','chair',
                   'window','door','table','book','phone','cup','bag','umbrella',
                   'bicycle','lamp','clock','flower','shoe',
                   'hat','computer','keyboard','mouse','camera',
                   'bottle','plant','mirror','sofa','desk'] + ['obj'+str(i) for i in range(19)]
objects_absent  = ['airplane','boat','horse','elephant','zebra','giraffe',
                   'tennis_racket','skateboard','surfboard','pizza','cake',
                   'fire_hydrant','bench','bus','truck','train','kite',
                   'baseball_bat','frisbee','snowboard'] + ['absent'+str(i) for i in range(30)]

grounded_model = POPEEvaluator()
hallucinator   = POPEEvaluator()

for obj in objects_present[:50]:
    grounded_model.add_question(obj, True, 'Yes, there is a ' + obj)
    hallucinator.add_question(obj, True, 'Yes, there is a ' + obj)

for obj in objects_absent[:50]:
    # Grounded model mostly says no
    grounded_resp = 'No' if np.random.rand() > 0.08 else 'Yes'
    # Hallucinating model frequently says yes to absent objects
    hall_resp = 'No' if np.random.rand() > 0.35 else 'Yes'
    grounded_model.add_question(obj, False, grounded_resp)
    hallucinator.add_question(obj, False, hall_resp)

print('POPE Evaluation Results:')
print(f'{"Metric":<22} {"Grounded Model":>16} {"Hallucinator":>14}')
gm = grounded_model.evaluate()
hm = hallucinator.evaluate()
for key in ['accuracy','precision','recall','f1','hallucination_rate','n_hallucinations']:
    print(f'  {key:<20}  {str(gm[key]):>14}  {str(hm[key]):>14)')

In [ ]:
# Multimodal benchmark scoring — simulated MMBench categories
import numpy as np

benchmark_categories = [
    'Object Recognition',
    'Attribute Recognition',
    'Spatial Relationship',
    'Scene Understanding',
    'Counting',
    'Visual Reasoning',
    'OCR',
    'Chart/Figure Understanding',
    'Celebrity Recognition',
    'Commonsense Reasoning',
]

# Simulated performance: stronger model scores higher across all categories
np.random.seed(10)
model_a_scores = np.clip(np.random.normal(0.78, 0.08, len(benchmark_categories)), 0.5, 0.98)
model_b_scores = np.clip(np.random.normal(0.65, 0.10, len(benchmark_categories)), 0.4, 0.90)

print('MMBench-style Evaluation:')
print(f'{"Category":<30} {"Model A":>10} {"Model B":>10}')
print('-'*52)
for cat, a, b in zip(benchmark_categories, model_a_scores, model_b_scores):
    winner = '<-- A wins' if a > b else '  B wins -->'
    print(f'{cat:<30} {a:>10.3f} {b:>10.3f}  {winner}')
print('-'*52)
print(f'{"Overall":<30} {model_a_scores.mean():>10.3f} {model_b_scores.mean():>10.3f}')